# DEBUG - Finding Icon

- setup the correct paths for both the ```source_image``` and the ```icon_path```. 

> If you are using the resized images, make sure you import the correct image

> Make sure you close the opened window that displayed the image and the location of the icon, otherwise the kernel will crash and restart.

In [1]:
source_image_path = "../captures/r_QBKS_screenshot_0.png"
icon_image_path = "../data/icons/topup/icon-choose-mobile-data-package-version-1.jpg"

In [2]:
import cv2
import numpy as np

def find_center(kp1, kp2, matches, img_shape):
    '''
    Find the center of the matched icon in the source image using keypoints and matches.
    '''
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    center_x = int(np.mean(src_pts[:, 0, 0]))
    center_y = int(np.mean(src_pts[:, 0, 1]))

    top_left = (int(np.min(src_pts[:, 0, 0])), int(np.min(src_pts[:, 0, 1])))
    bottom_right = (int(np.max(src_pts[:, 0, 0])), int(np.max(src_pts[:, 0, 1])))

    return [center_x, center_y], top_left, bottom_right

def feature_based_matching(page_img, icon_img, gray_scale=False, method='SIFT'):
    '''
    Perform feature-based matching using SIFT or ORB.
    '''
    if gray_scale:
        page_img = cv2.cvtColor(page_img, cv2.COLOR_BGR2GRAY)
        icon_img = cv2.cvtColor(icon_img, cv2.COLOR_BGR2GRAY)

    if method == 'SIFT':
        detector = cv2.SIFT_create()
    elif method == 'ORB':
        detector = cv2.ORB_create()
    else:
        raise ValueError("Invalid method. Use 'SIFT' or 'ORB'.")

    kp1, des1 = detector.detectAndCompute(page_img, None)
    kp2, des2 = detector.detectAndCompute(icon_img, None)

    if method == 'SIFT':
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)
    else:
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

    matches = matcher.match(des1, des2)

    matches = sorted(matches, key=lambda x: x.distance)

    return kp1, kp2, matches

def find_icon(page_img_paths, icon_img_path, gray_scale=False, method='SIFT', num_matches=10, debug=True):
    '''
    Find the icon in a list of page images using feature-based matching.
    '''
    best = {'best_solution': None, 'best_solution_index': 0, 'loc': [0, 0], 'matches': []}

    for i, page_img_path in enumerate(page_img_paths):
        page_img = cv2.imread(page_img_path)
        icon_img = cv2.imread(icon_img_path)

        kp1, kp2, matches = feature_based_matching(page_img, icon_img, gray_scale, method)

        num_matches = min(len(matches), num_matches)
        match_quality = np.mean([m.distance for m in matches[:num_matches]])

        if best['best_solution'] is None or match_quality < best['best_solution']:
            best['best_solution'] = match_quality
            best['best_solution_index'] = i
            best['loc'], best['top_left'], best['bottom_right'] = find_center(kp1, kp2, matches[:num_matches], page_img.shape)
            best['matches'] = matches[:num_matches] 
            best_page_img = page_img.copy()

    if debug:
        print(f"Best match found in image index: {best['best_solution_index']}")
        print(f"Match quality (average distance): {best['best_solution']}")
        print(f"Number of matches used: {len(best['matches'])}")
        print(f"Center coordinates: {best['loc']}")
        print(f"Top-left corner: {best['top_left']}")
        print(f"Bottom-right corner: {best['bottom_right']}")

        matched_img = cv2.drawMatches(
            best_page_img, kp1, icon_img, kp2, best['matches'], None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )
        cv2.rectangle(best_page_img, best['top_left'], best['bottom_right'], (0, 0, 255), 2)
        cv2.imshow("Best Match", matched_img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    return best['loc'][0], best['loc'][1], best['best_solution_index']

# Example usage
find_icon([source_image_path], icon_image_path, gray_scale=False, method='SIFT', num_matches=20)

Best match found in image index: 0
Match quality (average distance): 47.71089668273926
Number of matches used: 20
Center coordinates: [1254, 499]
Top-left corner: (926, 369)
Bottom-right corner: (1311, 847)


(1254, 499, 0)

# DEBUG - Solving Captcha

- setup the correct paths for the ```captcha_image```.

> If you are using the resized images, make sure you import the correct image

> The found captcha will be printed out in the output console
> The entire processed image will be saved here!


In [ ]:
captcha_image = ""

In [ ]:
import easyocr
import cv2
import numpy as np

def extract_texts(image_path):
    reader = easyocr.Reader(['en'])
    img = cv2.imread(image_path)
    result = reader.readtext(img)
    return result

def solve_captcha(image_path, expected_length=5, save_for_debug=True, loc_finding=False):
    result = extract_texts(image_path)
    img = cv2.imread(image_path)
    img_with_boxes = img.copy()
    for detection in result:
        box = detection[0]
        text = detection[1]
        confidence = detection[2]
        pts = np.array(box, dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.polylines(img_with_boxes, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
        cv2.putText(img_with_boxes, text, (pts[0][0][0], pts[0][0][1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    if save_for_debug:
        output_path = 'captcha_image.png'
        cv2.imwrite(output_path, img_with_boxes)

    for detection in result:
        if loc_finding:
            print(f"Detected text: {detection[1]}, Coordinates: {detection[0]}")
        if len(detection[1]) == expected_length and detection[1].isdigit():
                return detection[1]
    return None


solved_captcha = solve_captcha(captcha_image)
print(solved_captcha)